# Image Reconstruction

- **Dataset**: UTKFace (cropped and aligned faces)
- **Encoder**: SHWSBic (Lp pooled bispectral scattering coefficients) vs Morlet wavelet scattering (from Kymatio)
- **Decoder**: Transposed convolutional layers
- **Loss**: Combined MSE, L1, and SSIM loss (0.5 * MSE + 0.2 * L1 + 0.3 * SSIM)


In [ ]:
import os
import sys
sys.path.append('..')

import cv2
import h5py
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torchmetrics
from torch.utils.data import DataLoader, Dataset, random_split
from tqdm import tqdm

from models import ScatteringAutoencoder
from scripts.train_autoencoder import (
    ScatteredFaceDataset, 
    get_dataloaders, 
    train_and_validate,
    plot_reconstructions
)
from wavelets import SolidHarmonicModel
from scattering.coefficients import calculate_coefficients

# Configuration
dim = 128
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")


## Prepare Dataset

First, we need to create an HDF5 dataset with pre-computed scattering coefficients. This can be done using the `prepare_reconstruction_dataset.py` script, or we can do it here interactively.


In [ ]:
scattering_config = {
    "L": 15,
    "J": 4,
    "sigma": 1.5,
    "lp_norms": [0.5, 1, 2, 4],
    "bispectrum": True,
    "bicoherence": True,
    "reduce": True
}

data_folder = "../datasets/UTKFace/"  # Update path
hdf5_path = f"../datasets/faces/utkface_scattering_{dim}x{dim}.h5"

# Uncomment to create dataset
# from scripts.prepare_reconstruction_dataset import create_hdf5_dataset
# create_hdf5_dataset(data_folder, hdf5_path, dim=dim, scattering_config=scattering_config)


## Load Dataset and Initialise Model


In [ ]:
dataset = ScatteredFaceDataset(hdf5_path)
print(f"Dataset size: {len(dataset)}")

scattering_dim = dataset[0][0].shape[-1]
print(f"Scattering coefficient dimension: {scattering_dim}")

model = ScatteringAutoencoder(scattering_dim).to(device)

batch_size = 64
train_loader, val_loader, test_loader = get_dataloaders(
    dataset, batch_size, train_split=0.8, val_split=0.1
)
print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")


## Train Autoencoder


In [ ]:
# Training configuration
num_epochs = 500
save_dir = "../models/reconstruction/checkpoints"
checkpoint_path = None  # Set to path if resuming from checkpoint

# Train model
best_val_loss, train_losses, val_losses = train_and_validate(
    model, train_loader, val_loader, num_epochs, device, 
    save_dir=save_dir, image_name="reconstruction", 
    checkpoint_path=checkpoint_path
)


## Visualize Reconstructions


In [ ]:
# Load best model
checkpoint = torch.load(f"{save_dir}/best_model.pth", map_location=device, weights_only=False)
model.load_state_dict(checkpoint['model_state_dict'])

# Plot reconstructions
plot_reconstructions(model, test_loader, device, num_samples=8, experiment_name="reconstruction", save_dir=save_dir)


## Learned Priors (Reconstructions from Noise)

In [ ]:
# Generate random noise images
num_samples = 9
noise_imgs = torch.rand((num_samples, dim, dim), device=device)

scattering = SolidHarmonicModel(
    M=dim, N=dim,
    scattering_config=scattering_config,
    higher_order_config=scattering_config.copy()
)
noise_scattered = scattering.scatter(noise_imgs)

model.eval()
with torch.no_grad():
    recon_from_noise = model(noise_scattered)

noise_imgs_np = noise_imgs.cpu().numpy()
recon_np = recon_from_noise.cpu().numpy()

fig, axes = plt.subplots(2, num_samples, figsize=(num_samples * 2, 4))

for i in range(num_samples):
    axes[0, i].imshow(noise_imgs_np[i], cmap='gray')
    axes[0, i].axis("off")
    if i == 0:
        axes[0, i].set_title('Noise')

    axes[1, i].imshow(recon_np[i], cmap='gray')
    axes[1, i].axis("off")
    if i == 0:
        axes[1, i].set_title('Reconstruction')

plt.tight_layout()
plt.savefig(f"{save_dir}/images/learned_priors.png", dpi=150, bbox_inches='tight')
plt.show()


## Latent Space Interpolation

In [ ]:
def get_sample_from_loader(loader, idx, device):
    batch_size = loader.batch_size
    batch_idx = idx // batch_size
    local_idx = idx % batch_size

    for i, (scattered_batch, images_batch) in enumerate(loader):
        if i == batch_idx:
            img = images_batch[local_idx:local_idx+1].to(device)
            z = scattered_batch[local_idx:local_idx+1].to(device)
            return img, z
    return None, None

idx1, idx2 = 0, 176
img1, z1 = get_sample_from_loader(test_loader, idx1, device)
img2, z2 = get_sample_from_loader(test_loader, idx2, device)

num_steps = 7
alphas = torch.linspace(0, 1, num_steps, device=device)
z_interp = [(1 - a) * z1 + a * z2 for a in alphas]
z_interp = torch.cat(z_interp, dim=0)

# Generate interpolated reconstructions
model.eval()
with torch.no_grad():
    recon_interp = model(z_interp)

recon_images = [img1.cpu().squeeze().numpy()] + \
               [recon_interp[i].cpu().squeeze().numpy() for i in range(num_steps)] + \
               [img2.cpu().squeeze().numpy()]

total_steps = len(recon_images)
fig, axes = plt.subplots(1, total_steps, figsize=(total_steps * 2, 2))

for i, img in enumerate(recon_images):
    axes[i].imshow(img, cmap='gray')
    axes[i].axis('off')

axes[0].set_title('Original 1')
axes[-1].set_title('Original 2')
plt.tight_layout()
plt.savefig(f"{save_dir}/images/interpolation.png", dpi=150, bbox_inches='tight')
plt.show()
